### Objetivo

Desenvolver e comparar modelos de Machine Learning
para identificar a presença de doenças cardiovasculares
a partir de características clínicas dos pacientes.

O projeto utiliza um conjunto de dados com aproximadamente
70.000 pacientes e contempla análise exploratória,
pré-processamento, engenharia de atributos, treinamento,
avaliação e interpretação dos modelos.

### Organização do notebook

1. Carregamento dos dados
2. Análise exploratória (EDA)
3. Pré-processamento e engenharia de atributos
4. Comparação dos modelos na validação
5. Avaliação final e limiar de decisão
6. Interpretação dos resultados (SHAP)
7. Conclusão

In [1]:
import sys
from pathlib import Path

# Permite rodar o notebook fora do Docker (no Docker o PYTHONPATH já aponta para src/)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from cardio.config import DATA_PATH
from cardio.data.loader import load_dataset
from cardio.data.preprocessing import clean, load_splits
from cardio.features.build import add_features, get_feature_names
import cardio.visualization.eda as eda

In [2]:

df = load_dataset(DATA_PATH)
df_clean = clean(df, verbose=True)
eda.plot_dados_invalidos(df_clean)

df_features = add_features(df_clean)

df_features[
    ["ap_hi", "ap_lo", "categoria_pressao", "imc", "pressao_pulso", "pam"]
].head(10)

features_a = get_feature_names("A")
features_b = get_feature_names("B")

print(f"Conjunto A: {len(features_a)} atributos")
print(f"Conjunto B: {len(features_b)} atributos")


X_train, X_val, X_test, y_train, y_val, y_test = load_splits(
    feature_set="A",
    save_ids=True,
    verbose=True
)


70.000 → 68.509 linhas (1.491 removidas, 2,13%)
  - Duplicados (ignorando o id): 24
  - Sistólica fora de 70–250 mmHg: 229
  - Diastólica fora de 40–200 mmHg: 997
  - Diastólica >= sistólica: 103
  - Altura fora de 130–210 cm: 89
  - Peso fora de 35–200 kg: 16
  - IMC fora de 15–60 kg/m²: 33
Taxa de doença: 49,97% → 49,49%
Conjunto A: 11 atributos
Conjunto B: 16 atributos
70.000 → 68.509 linhas (1.491 removidas, 2,13%)
  - Duplicados (ignorando o id): 24
  - Sistólica fora de 70–250 mmHg: 229
  - Diastólica fora de 40–200 mmHg: 997
  - Diastólica >= sistólica: 103
  - Altura fora de 130–210 cm: 89
  - Peso fora de 35–200 kg: 16
  - IMC fora de 15–60 kg/m²: 33
Taxa de doença: 49,97% → 49,49%
Treino: 47.955 pacientes (70,00%), 49,48% com doença
Validação: 10.277 pacientes (15,00%), 49,49% com doença
Teste: 10.277 pacientes (15,00%), 49,49% com doença


Treino: 47.955 pacientes (70,00%), 49,48% com doença
Validação: 10.277 pacientes (15,00%), 49,49% com doença
Teste: 10.277 pacientes (15,00%), 49,49% com doença


## 2. Análise Exploratória dos Dados (EDA)

A análise exploratória tem como objetivo compreender as características dos pacientes, identificar padrões relacionados à presença de doenças cardiovasculares e investigar possíveis inconsistências nos dados.

Inicialmente, avaliamos a distribuição da variável-alvo `cardio`, que indica a ausência (`0`) ou presença (`1`) de doença cardiovascular.

Em seguida, analisamos a distribuição etária, as correlações entre os atributos e os valores de pressão arterial sistólica e diastólica.

A etapa de limpeza também permite identificar registros inconsistentes, como valores de pressão arterial fora de limites plausíveis, que poderiam comprometer a qualidade do treinamento dos modelos.

### Distribuição da variável-alvo

A variável `cardio` representa a classificação utilizada no projeto:

- **0:** paciente sem doença cardiovascular registrada.
- **1:** paciente com doença cardiovascular registrada.

O gráfico permite observar a proporção de pacientes em cada classe após a limpeza dos dados.

Essa análise é importante porque um desequilíbrio acentuado entre as classes poderia influenciar o treinamento e a avaliação dos modelos de Machine Learning.

Por esse motivo, o projeto utiliza uma divisão estratificada dos dados em treinamento, validação e teste, preservando aproximadamente a proporção das classes em cada conjunto.

In [3]:
eda.plot_distribuicao_cardio(df_clean)

### Distribuição etária dos pacientes

A idade é uma variável importante na análise de risco cardiovascular, pois a ocorrência de doenças cardiovasculares pode variar entre diferentes faixas etárias.

O gráfico apresenta a distribuição das idades dos pacientes após a etapa de limpeza dos dados.

Essa visualização permite compreender o perfil etário da população estudada e identificar possíveis concentrações de pacientes em determinadas faixas de idade.

Posteriormente, essa variável será utilizada como atributo preditivo nos modelos de Machine Learning.

In [4]:
eda.plot_distribuicao_idade(df_clean)


### Matriz de correlação entre os atributos

A matriz de correlação permite identificar relações lineares entre as variáveis clínicas e demográficas presentes no conjunto de dados.

Os coeficientes variam de -1 a +1:

- **Próximo de +1:** correlação positiva forte.
- **Próximo de -1:** correlação negativa forte.
- **Próximo de 0:** ausência de correlação linear significativa.

Essa análise auxilia na identificação de atributos relacionados à variável-alvo `cardio` e de possíveis redundâncias entre as variáveis preditoras.

É importante destacar que correlação não implica causalidade. Além disso, atributos com baixa correlação individual ainda podem contribuir para o desempenho dos modelos de Machine Learning.

In [5]:
eda.plot_matriz_correlacao(df_clean)


### Distribuição da pressão arterial

A pressão arterial é um dos principais indicadores clínicos utilizados na avaliação da saúde cardiovascular.

Neste conjunto de dados, analisamos duas variáveis:

- **`ap_hi`:** pressão arterial sistólica, correspondente à pressão máxima durante a contração do coração.
- **`ap_lo`:** pressão arterial diastólica, correspondente à pressão mínima durante o relaxamento do coração.

O gráfico apresenta a distribuição dessas medidas após a aplicação das regras de limpeza.

Durante o pré-processamento, foram tratados registros com valores inconsistentes, como pressões negativas, valores fisiologicamente implausíveis e casos em que a pressão diastólica era maior ou igual à sistólica.

Essa etapa contribui para reduzir a influência de dados inválidos no treinamento dos modelos preditivos.

In [6]:
eda.plot_distribuicao_pressao_arterial(df_clean)

## 3. Pré-processamento e Engenharia de Atributos

O pré-processamento tem como objetivo melhorar a qualidade dos dados antes de sua utilização nos modelos de Machine Learning.

Nesta etapa, são aplicadas regras de limpeza para remover registros inconsistentes e são construídos novos atributos a partir das informações clínicas disponíveis.

### Limpeza dos dados

A função `clean()`, implementada no módulo `cardio.data.preprocessing`, centraliza as regras de limpeza utilizadas no projeto.

Entre as verificações realizadas estão a identificação de valores inválidos de pressão arterial e a exclusão de registros que não atendem aos critérios de consistência definidos.

O processo de limpeza é executado antes da engenharia de atributos, evitando que valores inconsistentes sejam utilizados na criação de novas variáveis.

### Engenharia de atributos

A função `add_features()`, implementada no módulo `cardio.features.build`, permite criar atributos derivados das informações originais.

Entre os atributos apresentados no projeto estão:

- **IMC:** índice de massa corporal, calculado a partir do peso e da altura.
- **Pressão de pulso:** diferença entre as pressões sistólica e diastólica.
- **PAM:** pressão arterial média estimada.
- **Categoria de pressão arterial:** classificação derivada dos valores de pressão.

Esses atributos podem fornecer informações complementares aos modelos, permitindo avaliar se a engenharia de atributos melhora o desempenho preditivo.

In [7]:
colunas_engenharia = [
    "ap_hi",
    "ap_lo",
    "categoria_pressao",
    "imc",
    "pressao_pulso",
    "pam",
]

display(df_features[colunas_engenharia].head(10))

,ap_hi,ap_lo,categoria_pressao,imc,pressao_pulso,pam
0,110,80,Hipertensão 1,21.967120,30,90.000000
1,140,90,Hipertensão 2,34.927679,50,106.666667
2,130,70,Hipertensão 1,23.507805,60,90.000000
3,150,100,Hipertensão 2,28.710479,50,116.666667
4,100,60,Normal,23.011177,40,73.333333
5,120,80,Hipertensão 1,29.384676,40,93.333333
6,130,80,Hipertensão 1,37.729725,50,96.666667
7,130,90,Hipertensão 2,29.983588,40,103.333333
8,110,70,Normal,28.440955,40,83.333333
9,110,60,Normal,25.282570,50,76.666667


### Comparação dos conjuntos de atributos A e B

Para investigar o impacto da engenharia de atributos no desempenho dos modelos, foram definidos dois conjuntos de variáveis preditoras.

**Conjunto A — Atributos originais**

Contém 11 atributos clínicos, demográficos e comportamentais disponíveis após a preparação inicial dos dados:

- Idade, sexo, altura e peso.
- Pressão arterial sistólica e diastólica.
- Colesterol e glicose.
- Tabagismo, consumo de álcool e atividade física.

**Conjunto B — Atributos originais e derivados**

Contém os 11 atributos do Conjunto A, acrescidos de cinco variáveis construídas pela engenharia de atributos:

- `imc`: índice de massa corporal.
- `pressao_pulso`: diferença entre as pressões sistólica e diastólica.
- `pam`: pressão arterial média estimada.
- `categoria_pressao`: categoria derivada das medidas de pressão arterial.
- `faixa_imc`: classificação por faixa de IMC.

A comparação entre os conjuntos permite avaliar se os atributos derivados acrescentam informações úteis à capacidade preditiva dos modelos.

A escolha do conjunto final é baseada nos resultados obtidos durante a validação, sem utilizar o conjunto de teste para selecionar atributos ou ajustar os modelos.

In [8]:
print("CONJUNTO A")
print(f"Total: {len(features_a)} atributos")
print(features_a)

print("\nCONJUNTO B")
print(f"Total: {len(features_b)} atributos")
print(features_b)

CONJUNTO A
Total: 11 atributos
['age_years', 'sex_male', 'height', 'weight', 'ap_hi', 'ap_lo', 'cholesterol', 'gluc', 'smoke', 'alco', 'active']

CONJUNTO B
Total: 16 atributos
['age_years', 'sex_male', 'height', 'weight', 'ap_hi', 'ap_lo', 'cholesterol', 'gluc', 'smoke', 'alco', 'active', 'imc', 'pressao_pulso', 'pam', 'categoria_pressao', 'faixa_imc']


## 4. Comparação dos Modelos na Validação

Nesta etapa, são apresentados os resultados dos modelos de Machine Learning avaliados durante o desenvolvimento do projeto.

Foram considerados dois conjuntos de atributos: A, composto pelas 11 variáveis originais, e B, que inclui cinco atributos derivados adicionais.

A comparação utiliza métricas calculadas sobre o conjunto de validação, permitindo avaliar o desempenho dos algoritmos e selecionar a configuração mais adequada para o problema cardiovascular.

Os resultados foram previamente calculados e armazenados em arquivos CSV, evitando a necessidade de treinar novamente os modelos durante a execução deste notebook.

In [9]:
import pandas as pd
from pathlib import Path

NOMES = {"xgb": "XGBoost", "svm": "SVM", "random_forest": "Random Forest", "knn": "KNN", "logistic": "Regressão Logística",
         "baseline": "Baseline (sorteio)"}

# Resultado mais recente de cada modelo e conjunto na validação (limiar padrão 0,50), com o baseline da T23.
val = pd.read_csv(Path("../results/val_results.csv")).drop_duplicates(["modelo", "features"], keep="last")
baseline = pd.read_csv(Path("../results/comparison_table.csv")).query("modelo == 'baseline'")
colunas = ["modelo", "features", "acuracia", "precision_score", "recall_score", "f1_score", "auc_roc"]
df_comparison = pd.concat([val[colunas], baseline[colunas]], ignore_index=True)

df_a = df_comparison[df_comparison["features"] == "A"].set_index("modelo")
df_b = df_comparison[df_comparison["features"] == "B"].set_index("modelo")
tabela = df_a.sort_values("auc_roc", ascending=False)[["acuracia", "precision_score", "recall_score", "f1_score", "auc_roc"]]
tabela["auc_roc_conjunto_B"] = df_b["auc_roc"]
tabela.index = [NOMES[m] for m in tabela.index]
display(tabela.round(4))

,acuracia,precision_score,recall_score,f1_score,auc_roc,auc_roc_conjunto_B
XGBoost,0.7263,0.7453,0.6789,0.7106,0.7919,0.7916
SVM,0.7283,0.7411,0.6933,0.7164,0.7915,0.7911
Random Forest,0.7286,0.7523,0.6734,0.7107,0.7908,0.7896
KNN,0.7249,0.7534,0.6602,0.7038,0.7853,0.7862
Regressão Logística,0.7267,0.7503,0.6711,0.7085,0.7831,0.7851
Baseline (sorteio),0.5086,0.5036,0.4943,0.4989,0.5085,0.5085


In [10]:
# Limiar ajustado (T22): recall mínimo de 0,80 na validação. Resultado do modelo escolhido, do comparison_table.csv.
ajustado = pd.read_csv(Path("../results/comparison_table.csv")).query("modelo == 'xgb' and features == 'A'")
display(ajustado[["modelo", "features", "auc_roc", "limiar", "recall_ajustado", "precision_ajustada", "f1_ajustado",
                  "doentes_nao_sinalizados_por_1000", "saudaveis_encaminhados_por_1000"]].round(4).reset_index(drop=True))

,modelo,features,auc_roc,limiar,recall_ajustado,precision_ajustada,f1_ajustado,doentes_nao_sinalizados_por_1000,saudaveis_encaminhados_por_1000
0,xgb,A,0.7919,0.36,0.8169,0.6581,0.7289,90.5906,210.0808


### Análise comparativa e seleção do modelo final

Foram comparados cinco algoritmos: Regressão Logística, K-Nearest Neighbors (KNN), Random Forest, XGBoost e Support Vector Machine (SVM), utilizando os conjuntos de atributos A e B.

A AUC-ROC dos cinco modelos ficou entre **0,783 e 0,792** (contra 0,508 do baseline, que sorteia a classe): todos aprendem um padrão real, mas a diferença entre eles é pequena. XGBoost, SVM e Random Forest empataram na prática; o XGBoost foi escolhido por ter a maior AUC, treinar em menos de um segundo e permitir explicar as decisões com o SHAP.

O **XGBoost com o conjunto A** apresentou o melhor desempenho global na validação, considerando os critérios estabelecidos para o projeto.

Os principais resultados foram:

- **AUC-ROC:** 0,7919.
- **Recall ajustado:** 81,69%.
- **Precisão ajustada:** 65,81%.
- **F1-score ajustado:** 0,7289.
- **Limiar de decisão:** 0,36.

A escolha priorizou a capacidade de identificar pacientes com doença cardiovascular, buscando reduzir a ocorrência de falsos negativos.

Embora o conjunto B tenha incorporado cinco atributos derivados, ele não apresentou melhoria no desempenho do XGBoost em relação ao conjunto A.

Assim, o **XGBoost com 11 atributos** foi selecionado como modelo final, utilizando o limiar de decisão de 0,36 definido na etapa de validação.

Essa seleção foi realizada sem utilizar os resultados do conjunto de teste, preservando sua finalidade de avaliação final independente.

### Curvas ROC, Precision-Recall e calibração

As curvas dos cinco modelos praticamente se sobrepõem e ficam bem acima da diagonal do acaso. Na curva Precision-Recall, a precisão média vai de 0,761 (Regressão Logística) a 0,775 (XGBoost).
A calibração do XGBoost acompanha a diagonal (desvio médio de 1,4 ponto percentual), então a probabilidade prevista pode ser lida como uma estimativa de risco.

![Curvas ROC dos modelos](../results/figures/roc_A.png)

![Curvas Precision-Recall dos modelos](../results/figures/pr_A.png)

![Calibração do XGBoost](../results/figures/calibration_xgb.png)

## 5. Avaliação Final do Modelo

Após a comparação dos algoritmos na validação, foi selecionado o XGBoost com o conjunto A de atributos e limiar de decisão de 0,36.

O modelo final foi treinado utilizando os conjuntos de treinamento e validação, conforme a metodologia definida no projeto.

Nesta seção, carregamos o modelo final previamente salvo e apresentamos os resultados obtidos no conjunto de teste independente.

O notebook não realiza novos treinamentos nem recalcula as métricas de teste, preservando os resultados da avaliação final já realizada.

In [11]:
import joblib

model_path = Path("../models/final_model.joblib")

final_artifact = joblib.load(model_path)

print("Modelo:", final_artifact["modelo"])
print("Conjunto de atributos:", final_artifact["features"])
print("Limiar de decisão:", final_artifact["threshold"])
print("Pipeline carregado:", type(final_artifact["model"]).__name__)

Modelo: xgb
Conjunto de atributos: A
Limiar de decisão: 0.36
Pipeline carregado: Pipeline


In [12]:
test_results_path = Path("../results/test_results.csv")

df_test_results = pd.read_csv(test_results_path)

display(df_test_results)

,modelo,features,n_pacientes,limiar,accuracy,auc_roc,recall,precision,f1,especificidade,doentes_nao_sinalizados_por_1000,saudaveis_encaminhados_por_1000
0,xgb,A,10277,0.36,0.713243,0.809812,0.831302,0.669305,0.741559,0.597573,83.487399,203.269437


### Resultados e interpretação da avaliação final

O modelo XGBoost, utilizando o conjunto A de atributos e limiar de decisão de 0,36, foi avaliado em **10.277 pacientes** do conjunto de teste independente.

Os resultados demonstraram:

- **AUC-ROC de 0,8098:** indica a capacidade do modelo de discriminar pacientes com e sem doença cardiovascular.
- **Recall de 83,13%:** aproximadamente 83 em cada 100 pacientes com doença cardiovascular foram corretamente identificados.
- **Precisão de 66,93%:** entre os pacientes classificados como positivos, aproximadamente 67% realmente pertenciam à classe positiva.
- **F1-score de 0,7416:** representa o equilíbrio entre precisão e recall.
- **Especificidade de 59,76%:** aproximadamente 60% dos pacientes sem doença cardiovascular foram corretamente classificados como negativos.

Considerando a população total avaliada, foram observados aproximadamente **83 pacientes com doença cardiovascular não sinalizados a cada 1.000 registros**, além de **203 pacientes sem doença cardiovascular classificados como positivos a cada 1.000 registros**.

O recall obtido superou a meta de 80% estabelecida durante a validação.

Entretanto, o modelo ainda apresenta falsos negativos e falsos positivos. Por isso, seus resultados devem ser interpretados como apoio à identificação de risco cardiovascular, e não como diagnóstico médico definitivo.

A avaliação no conjunto de teste foi realizada após a seleção do modelo e do limiar na validação, sem novos ajustes baseados nos resultados finais.

### Matriz de confusão e limiar de decisão

Com o limiar de 0,36, dos 5.086 pacientes com doença no teste, o modelo sinalizou **4.228** e deixou de sinalizar **858**; dos 5.191 sem doença, liberou 3.102 e encaminhou **2.089** sem necessidade.
A matriz abaixo foi reconstruída a partir das taxas de `results/test_results.csv` (o modelo não foi avaliado de novo no teste).

![Matriz de confusão do modelo final no teste](../results/figures/t33_matriz_confusao_teste.png)

O limiar de 0,36 é o maior que mantém o recall em pelo menos 0,80 na validação (regra da T22). Na validação, ele reduz os doentes não sinalizados de 159 para 91 a cada 1.000 pacientes,
ao custo de 95 encaminhamentos extras de pessoas saudáveis.

![Recall, precision e F1 por limiar de decisão, na validação](../results/figures/t33_limiar_xgb_A.png)

### Onde o modelo erra mais (T26)

A análise de erros usa a **validação** e o limiar final (0,36). O recall geral é 81,7%, mas varia muito entre os subgrupos:

- **Sexo:** praticamente igual (82,0% nas mulheres e 81,3% nos homens).
- **Idade:** 58,8% abaixo de 45 anos, 69,0% entre 45 e 55 anos e 93,9% acima de 55 anos.
- **Pressão arterial:** 30,1% com pressão normal, 50,0% com pressão elevada, 68,9% na hipertensão 1 e 99,3% na hipertensão 2.

Dos 5.086 doentes da validação, **929 (18%) não foram sinalizados**. Entre eles, 27% têm pressão normal (contra 3% dos doentes que o modelo encontrou) e 30% não têm nenhum fator de risco aparente (contra 11%).
Ou seja, o modelo erra principalmente em doentes que, nas variáveis disponíveis, parecem saudáveis: é uma limitação da informação, e não só do modelo. Por isso um resultado "sem doença" **não exclui** o risco, e a decisão final é sempre do médico.

![Desempenho por subgrupo](../results/figures/t26_subgrupos.png)

![Falsos negativos comparados aos demais pacientes](../results/figures/t26_falsos_negativos.png)

## 6. Interpretabilidade do Modelo

A interpretabilidade permite compreender quais características influenciam as previsões do XGBoost selecionado como modelo final.

### Importância das variáveis (T24)

A importância por permutação embaralha os valores de uma variável na validação e mede o quanto a AUC cai. Calculada para a Regressão Logística, o Random Forest e o XGBoost, os **três modelos concordam** nas três variáveis mais importantes e na ordem: **pressão sistólica, idade e colesterol**. Embaralhar a pressão sistólica derruba a AUC em 0,16, de cinco a seis vezes o efeito da idade e do colesterol. Na Regressão Logística, cada 10 mmHg a mais na pressão sistólica multiplica a chance de doença por 1,75.

![Importância por permutação nos três modelos](../results/figures/t24_permutacao.png)

### Análise global com SHAP

A análise SHAP foi realizada em uma amostra de 1.500 pacientes. Os valores SHAP representam contribuições na escala interna do modelo (*log-odds*), não alterações diretas na probabilidade de doença.

As três características mais importantes foram **pressão arterial sistólica, idade e colesterol**, responsáveis por aproximadamente **76% da importância global**.

![Resumo SHAP](../results/figures/t25_shap_summary.png)

### Relação entre os atributos e as previsões

A contribuição da pressão sistólica para a previsão de doença aumenta a partir de aproximadamente **130 mmHg**. Para a idade, a contribuição passa a favorecer a classificação de doença a partir de aproximadamente **55 anos**.

![SHAP - Pressão sistólica](../results/figures/t25_shap_dependencia_pressao.png)

![SHAP - Idade](../results/figures/t25_shap_dependencia_idade.png)

Os hábitos autodeclarados, como tabagismo, consumo de álcool e atividade física, apresentaram menor importância nas previsões.

### Explicações individuais

Os gráficos *waterfall* mostram como cada atributo contribuiu para a classificação de um paciente específico.

**Paciente com doença corretamente identificado:**

![SHAP - Acerto com doença](../results/figures/t25_shap_waterfall_acerto_doente.png)

**Paciente sem doença corretamente identificado:**

![SHAP - Acerto sem doença](../results/figures/t25_shap_waterfall_acerto_saudavel.png)

**Paciente com doença não identificado pelo modelo (falso negativo):**

![SHAP - Falso negativo](../results/figures/t25_shap_waterfall_erro.png)

### Interpretação clínica e limitações

Os resultados mostram que o modelo utiliza principalmente variáveis clínicas relevantes para a classificação cardiovascular. Entretanto, a importância de uma variável não demonstra relação causal.

O SHAP explica o comportamento do modelo, mas não comprova sua segurança clínica. Como a amostra utilizada pertence aos dados de treino e validação do modelo final, essa análise não representa uma avaliação independente de desempenho.

O modelo deve ser considerado apenas uma ferramenta experimental de apoio à análise, não um substituto da avaliação médica.

## 7. Conclusão

O projeto teve como objetivo desenvolver e avaliar modelos de Machine Learning para identificar pacientes com possível presença de doença cardiovascular, utilizando informações clínicas, demográficas e comportamentais.

Durante o desenvolvimento, foram realizadas etapas de análise exploratória, limpeza dos dados, engenharia de atributos, treinamento, comparação de algoritmos e avaliação final.

### Principais resultados

Foram comparados cinco algoritmos: Regressão Logística, K-Nearest Neighbors (KNN), Random Forest, XGBoost e Support Vector Machine (SVM), utilizando dois conjuntos de atributos.

O **XGBoost com o conjunto A**, composto por 11 atributos, apresentou o melhor desempenho na validação e foi selecionado como modelo final.

Após o treinamento final e a avaliação em **10.277 pacientes do conjunto de teste**, foram obtidos os seguintes resultados:

- **AUC-ROC:** 0,8098.
- **Recall:** 83,13%.
- **Precisão:** 66,93%.
- **F1-score:** 0,7416.
- **Acurácia:** 71,32%.
- **Limiar de decisão:** 0,36.

O recall obtido superou a meta de 80%, o que significa que o modelo sinalizou a maior parte dos pacientes que de fato tinham doença cardiovascular no conjunto de teste.

### Por que o desempenho para perto de 0,79 de AUC?

Os cinco modelos chegaram ao mesmo patamar. Mais dados não ajudam (a AUC para de subir com cerca de um quarto do treino), modelos mais complexos pioram na validação (a AUC cai de 0,792 para 0,759 com árvores mais profundas, porque o modelo decora ruído) e as duas pressões arteriais sozinhas já dão AUC de 0,746. Isso indica que o limite está nas informações disponíveis, e não no algoritmo.

![Curva de aprendizado e efeito da complexidade do modelo](../results/figures/t33_limite_do_desempenho.png)

### Limitações e considerações finais

Apesar dos resultados, o modelo ainda apresenta falsos positivos e falsos negativos. Portanto, suas previsões não devem ser utilizadas isoladamente para estabelecer diagnósticos ou decisões clínicas.

Além disso, o desempenho observado no conjunto de teste não garante resultados equivalentes em populações diferentes, sendo necessárias validações adicionais antes de qualquer aplicação prática.

Conclui-se que o XGBoost apresentou resultados promissores para a tarefa de classificação cardiovascular estudada, demonstrando o potencial do Machine Learning como ferramenta de apoio à identificação de pacientes que podem necessitar de avaliação clínica adicional.